<a href="https://colab.research.google.com/github/Adhira-Deogade/pytorch-learnings/blob/main/Trash_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
!git lfs install

Git LFS initialized.


In [20]:
!git clone https://huggingface.co/datasets/vatsal-a/ZotBins

Cloning into 'ZotBins'...
remote: Enumerating objects: 610, done.
remote: Total 610 (delta 0), reused 0 (delta 0), pack-reused 610 (from 1)
Receiving objects: 100% (610/610), 85.40 KiB | 8.54 MiB/s, done.
Filtering content: 100% (600/600), 9.31 MiB | 960.00 KiB/s, done.


In [21]:
import os
import pandas as pd
from torchvision.io import decode_image

In [22]:
from torch.utils.data import Dataset, DataLoader

In [23]:
all_directories = os.listdir('ZotBins')
all_classes = [directory for directory in all_directories if not directory.startswith('.')]
print(all_classes)
# classes_dict = enumerate(all_classes)
classes_dict = {
    all_classes[i]: i for i in range(len(all_classes))
}
print(type(classes_dict))
print(classes_dict)
for k, v in classes_dict.items():
  print(k, v)
# # classes_dict = {
# #     classes[i]: i for i in range(len(classes))
# # }
# classes_dict = {}
# # for i in range(len(classes)):
# #   if classes[i].startswith('.'):
# #     continue
#   classes_dict[classes[i]] = i
# print(classes_dict)

['cardboard', 'glass', 'plastic', 'trash', 'metal', 'paper']
<class 'dict'>
{'cardboard': 0, 'glass': 1, 'plastic': 2, 'trash': 3, 'metal': 4, 'paper': 5}
cardboard 0
glass 1
plastic 2
trash 3
metal 4
paper 5


In [6]:
class CustomTrashClassificationDataset(Dataset):
  def __init__(self, img_dir, classes_dict, transforms=None, target_transforms=None, train=True):
    self.img_dir = img_dir
    self.classes = classes_dict.keys()
    self.len_classes = len(self.classes)
    self.transforms = transforms
    self.target_transforms = target_transforms
  def get_image_path(self, class_dir, idx):
    all_images_in_class = os.listdir(os.path.join(self.img_dir, class_dir))
  def __len__(self):
    return self.len_classes
  def __getitem__(self, idx):
    # for class_dir in self.classes:
    self.len_classes
    all_images_in_class = os.listdir(os.path.join(self.img_dir, class_dir))
    img_path = os.path.join(self.img_dir, class_dir, all_images_in_class[idx])
    print(img_path)
    # ZotBins/carboard/
    image = decode_image(img_path)
    label = classes_dict[class_dir]
    if self.transforms:
      image = self.transforms(image)
    if self.target_transforms:
      label = self.target_transforms(label)
    return image, label

In [7]:
# all_classes = os.listdir('ZotBins')
# print(len(all_classes))

In [8]:
# print(all_classes)

In [9]:
trash_dataset = CustomTrashClassificationDataset('ZotBins', classes_dict)

In [10]:
print(trash_dataset)

In [11]:
print(len(trash_dataset))

6


In [12]:
# from torch.utils.data import DataLoader

# train_dataloader = DataLoader(training_data, batch_size=64, shuffle=True)
# test_dataloader = DataLoader(test_data, batch_size=64, shuffle=True)

In [13]:
import torch
from torch.utils.data import random_split
generator1 = torch.Generator().manual_seed(42)
generator2 = torch.Generator().manual_seed(42)
print(random_split(range(10), [3, 7], generator=generator1))
# random_split(range(30), [0.3, 0.3, 0.4], generator=generator2)

[<torch.utils.data.dataset.Subset object at 0x78fc00824190>, <torch.utils.data.dataset.Subset object at 0x78fc00832050>]


In [14]:
train_data, test_data = random_split(trash_dataset, [0.8, 0.2], generator=generator1)

In [15]:
print(len(train_data))
print(len(test_data))

5
1


In [16]:
print(train_data[1])

NameError: name 'class_dir' is not defined

In [ ]:
print(len(trash_dataset))

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_data, batch_size=8, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=8, shuffle=True)

In [ ]:
print(len(train_dataloader))

In [ ]:
print(len(test_dataloader))

In [ ]:
print(len(train_data))

In [34]:
# I am trying to create a csv file in the format
# category, image name, numerical category(using classes_dict)
# The path of the image will be created with
# Root directory(ZotBins) + class directory(cardboard) + filename
# Reverse engineering CSV file:
# I first need to create a dictionary and dump dict to csv
# I am going to try to use polars library, because it's faster
# using multiple CPU cores at once
# Learn about polars here - https://blog.jetbrains.com/pycharm/2024/07/polars-vs-pandas/
# I am creating a dictionary with three
# Maybe I can create a tuple, that would be nice
# image, numeric class
import os
import polars as pl
from pprint import pprint
img_dir = 'ZotBins'
labelled_dict = {}
# labelled_dict = {
#     "column1": list1,
#     "column2": list2
# }
all_class_paths = []
all_categories = []
for class_dir in all_classes:
  numerical_category = classes_dict[class_dir]
  # print(numerical_category)
  class_paths = os.listdir(os.path.join(img_dir, class_dir))
  numerical_cat_array = [numerical_category] * len(class_paths)
  all_class_paths.extend(class_paths)
  all_categories.extend(numerical_cat_array)
  # tuple_image_class = dict(zip(class_paths, numerical_cat_array))
  # print(tuple_image_class)
  # labelled_dict.update(tuple_image_class)
# print(labelled_dict)
df = pl.DataFrame({"Image names": all_class_paths, "class names" :all_categories})
print(df)
df.write_csv('labelled_data.csv', include_header=False)
# for k, v in labelled_dict.items():
#   print(k, v)
  # tuple_image_class = tuple(zip(class_paths, numerical_cat_array))
  # print(tuple_image_class
  # all_class_paths.extend(tuple_image_class)
  # labelled_dict[numerical_category] = class_paths
# pprint(labelled_dict)

  # all_class_paths.extend(class_paths)
# print(f"Length of all class paths = {len(all_class_paths)}")
# for item in all_class_paths:
#   print(item)

shape: (600, 2)
┌──────────────────┬─────────────┐
│ Image names      ┆ class names │
│ ---              ┆ ---         │
│ str              ┆ i64         │
╞══════════════════╪═════════════╡
│ cardboard230.jpg ┆ 0           │
│ cardboard98.jpg  ┆ 0           │
│ cardboard219.jpg ┆ 0           │
│ cardboard181.jpg ┆ 0           │
│ cardboard184.jpg ┆ 0           │
│ …                ┆ …           │
│ paper520.jpg     ┆ 5           │
│ paper485.jpg     ┆ 5           │
│ paper297.jpg     ┆ 5           │
│ paper295.jpg     ┆ 5           │
│ paper446.jpg     ┆ 5           │
└──────────────────┴─────────────┘


In [28]:
import polars as pl

# # Create two lists of data
# list1 = [1, 2, 3, 4]
# list2 = ["A", "B", "C", "D"]

# # Create a Polars DataFrame from the lists, specifying column orientation as "col"
# # This will create a DataFrame with two columns
# df = pl.DataFrame({"column1": list1, "column2": list2}, orient="col")
# print(df)
# df2 = pl.DataFrame({"column1": list1, "column2": list2})
# print(df2)

# Write the DataFrame to a CSV file without including the header row
# df.write_csv("output.csv", include_header=False)

my_dict = {"key1": 10, "key2": 20, "key3": 30}
import polars as pl

df = pl.DataFrame({
    "Keys": my_dict.keys(),
    "Values": my_dict.values()
})


In [29]:
print(df)

shape: (3, 2)
┌──────┬────────┐
│ Keys ┆ Values │
│ ---  ┆ ---    │
│ str  ┆ i64    │
╞══════╪════════╡
│ key1 ┆ 10     │
│ key2 ┆ 20     │
│ key3 ┆ 30     │
└──────┴────────┘


In [36]:
df2 = pl.from_dict(labelled_dict)
print(type(df2))
print(df2)

<class 'polars.dataframe.frame.DataFrame'>
shape: (100, 6)
┌──────────────────┬──────────────┬────────────────┬──────────────┬──────────────┬──────────────┐
│ 0                ┆ 1            ┆ 2              ┆ 3            ┆ 4            ┆ 5            │
│ ---              ┆ ---          ┆ ---            ┆ ---          ┆ ---          ┆ ---          │
│ str              ┆ str          ┆ str            ┆ str          ┆ str          ┆ str          │
╞══════════════════╪══════════════╪════════════════╪══════════════╪══════════════╪══════════════╡
│ cardboard230.jpg ┆ glass176.jpg ┆ plastic186.jpg ┆ trash4.jpg   ┆ metal336.jpg ┆ paper332.jpg │
│ cardboard98.jpg  ┆ glass364.jpg ┆ plastic181.jpg ┆ trash10.jpg  ┆ metal41.jpg  ┆ paper118.jpg │
│ cardboard219.jpg ┆ glass439.jpg ┆ plastic354.jpg ┆ trash81.jpg  ┆ metal309.jpg ┆ paper73.jpg  │
│ cardboard181.jpg ┆ glass211.jpg ┆ plastic345.jpg ┆ trash44.jpg  ┆ metal268.jpg ┆ paper453.jpg │
│ cardboard184.jpg ┆ glass370.jpg ┆ plastic60.jpg  ┆ trash8

In [38]:
df.write_csv('labelled_data_numerclass.csv')

In [39]:
df_loaded = pl.read_csv('labelled_data_numerclass.csv')

In [40]:
print(df_loaded)

shape: (100, 6)
┌──────────────────┬──────────────┬────────────────┬──────────────┬──────────────┬──────────────┐
│ 0                ┆ 1            ┆ 2              ┆ 3            ┆ 4            ┆ 5            │
│ ---              ┆ ---          ┆ ---            ┆ ---          ┆ ---          ┆ ---          │
│ str              ┆ str          ┆ str            ┆ str          ┆ str          ┆ str          │
╞══════════════════╪══════════════╪════════════════╪══════════════╪══════════════╪══════════════╡
│ cardboard230.jpg ┆ glass176.jpg ┆ plastic186.jpg ┆ trash4.jpg   ┆ metal336.jpg ┆ paper332.jpg │
│ cardboard98.jpg  ┆ glass364.jpg ┆ plastic181.jpg ┆ trash10.jpg  ┆ metal41.jpg  ┆ paper118.jpg │
│ cardboard219.jpg ┆ glass439.jpg ┆ plastic354.jpg ┆ trash81.jpg  ┆ metal309.jpg ┆ paper73.jpg  │
│ cardboard181.jpg ┆ glass211.jpg ┆ plastic345.jpg ┆ trash44.jpg  ┆ metal268.jpg ┆ paper453.jpg │
│ cardboard184.jpg ┆ glass370.jpg ┆ plastic60.jpg  ┆ trash87.jpg  ┆ metal7.jpg   ┆ paper283.jpg │
│ … 

In [42]:
print(df_loaded['0'])

shape: (100,)
Series: '0' [str]
[
	"cardboard230.jpg"
	"cardboard98.jpg"
	"cardboard219.jpg"
	"cardboard181.jpg"
	"cardboard184.jpg"
	…
	"cardboard191.jpg"
	"cardboard156.jpg"
	"cardboard99.jpg"
	"cardboard208.jpg"
	"cardboard231.jpg"
]


In [ ]:
class CustomTrashClassificationDatasetNumber(Dataset):
  def __init__(self, img_dir, classes_dict, image_paths_df, transforms=None, target_transforms=None, train=True):
    self.img_dir = img_dir
    self.classes_dict = classes_dict
    self.file_paths = image_paths
    self.classes = classes_dict.keys()
    self.len_files = len(self.file_paths)
    self.transforms = transforms
    self.target_transforms = target_transforms
  def __len__(self):
    return self.len_files
  def __getitem__(self, idx):
    # for class_dir in self.classes:
    # all_images_in_class = os.listdir(os.path.join(self.img_dir, class_dir))
    # img_path = os.path.join(self.img_dir, class_dir, all_images_in_class[idx])
    # print(img_path)
    # ZotBins/carboard/
    image = decode_image(self.file_paths[idx])
    label = classes_dict[class_dir]
    if self.transforms:
      image = self.transforms(image)
    if self.target_transforms:
      label = self.target_transforms(label)
    return image, label